In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import joblib
import os
print("libraries loaded!")
print("tensorflow version:", tf.__version__)

libraries loaded!
tensorflow version: 2.21.0


In [3]:
df = pd.read_csv('../data/processed/weather_clean.csv')
print(df.shape)
print(df.columns.tolist())

C:\Users\SAMPATH\AppData\Local\Temp\ipykernel_32744\2124382371.py:1: DtypeWarning: Columns (0: station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/weather_clean.csv')


(27621770, 13)
['station_id', 'city_name', 'date', 'season', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'avg_wind_speed_kmh', 'month', 'day', 'year', 'will_rain']


In [4]:
city = 'Brussels'
df_city = df[df['city_name'] == city].copy()
df_city = df_city.sort_values(['year', 'month', 'day']).reset_index(drop=True)
print("rows for", city, ":", len(df_city))
print(df_city.head())

rows for Brussels : 69347
  station_id city_name        date  season  avg_temp_c  min_temp_c  \
0       6447  Brussels  1833-01-02  Winter        17.9        -4.8   
1       6447  Brussels  1833-01-03  Winter        17.9        -6.8   
2       6447  Brussels  1833-01-04  Winter        17.9        -6.8   
3       6447  Brussels  1833-01-05  Winter        17.9        -9.0   
4       6447  Brussels  1833-01-06  Winter        17.9        -8.5   

   max_temp_c  precipitation_mm  avg_wind_speed_kmh  month  day  year  \
0        -1.4               0.0                10.9      1    2  1833   
1        -3.1               0.0                10.9      1    3  1833   
2        -3.9               0.0                10.9      1    4  1833   
3        -4.4               0.0                10.9      1    5  1833   
4         1.4               0.0                10.9      1    6  1833   

   will_rain  
0          0  
1          0  
2          0  
3          0  
4          0  


In [5]:
features = ['avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'avg_wind_speed_kmh']
data = df_city[features].values
print("data shape:", data.shape)

data shape: (69347, 5)


In [6]:
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)
print("scaled shape:", data_scaled.shape)
print("min value:", data_scaled.min())
print("max value:", data_scaled.max())

scaled shape: (69347, 5)
min value: 0.0
max value: 1.0


In [7]:
def make_sequences(data, lookback=30, future=7):
    x, y = [], []
    for i in range(len(data) - lookback - future):
        x.append(data[i : i+lookback])
        y.append(data[i+lookback : i+lookback+future, 0])
    return np.array(x), np.array(y)
x, y = make_sequences(data_scaled)
print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: (69310, 30, 5)
y shape: (69310, 7)


In [8]:
split = int(len(x) * 0.8)
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]
print("train samples:", len(x_train))
print("test samples:", len(x_test))

train samples: 55448
test samples: 13862


In [9]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(64, return_sequences=True, input_shape=(30,5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(7)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary

c:\Users\SAMPATH\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


<bound method Model.summary of <Sequential name=sequential, built=True>>

In [10]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print("training done!")

Epoch 1/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step - loss: 0.0154 - mae: 0.0885 - val_loss: 0.0246 - val_mae: 0.1168
Epoch 2/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 0.0037 - mae: 0.0481 - val_loss: 0.0275 - val_mae: 0.1235
Epoch 3/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.0016 - mae: 0.0318 - val_loss: 0.0319 - val_mae: 0.1322
Epoch 4/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - loss: 6.8255e-04 - mae: 0.0205 - val_loss: 0.0331 - val_mae: 0.1345
Epoch 5/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 2.2370e-04 - mae: 0.0116 - val_loss: 0.0350 - val_mae: 0.1380
Epoch 6/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 5.2064e-05 - mae: 0.0055 - val_loss: 0.0363 - val_mae: 0.1407
Epoch 7/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - loss: 9.0214e-06 - mae: 0.0023 - val_loss: 0.0361 - val_mae: 0.1400
Epoch 8/10
780/780 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 2.2544e-06 - mae: 0.0012 - val_loss: 0.0360 - val_mae: 0.1399
Epoch 9/10
780/780 ━

In [12]:
os.makedirs('../models/lstm', exist_ok=True)
model.save('../models/lstm/model.h5')
joblib.dump(scaler, '../models/lstm/scaler.pkl')
print("lstm model saved!")

lstm model saved!
